<a href="https://colab.research.google.com/github/Manuel-Git-06/Curso-Conservacion-de-Ecosistemas/blob/main/PistasAudio.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
#@title Separador Vocal/Instrumental (MP4/MP3)
!pip install -q moviepy spleeter pydub

from google.colab import files
import os
from pathlib import Path
import traceback
from moviepy.editor import VideoFileClip, AudioFileClip
from spleeter.separator import Separator
import logging
from pydub import AudioSegment

# ================= Configuración =================
SPLEETER_MODEL = 'spleeter:2stems'  # Modelo a usar (puede ser '2stems', '4stems', etc.)
OUTPUT_DIR_NAME = "output_spleeter"  # Directorio donde se almacenarán los resultados
CONVERT_TO_MP3 = True              # Convertir la salida a MP3 si está en WAV
MEZCLAR_VOCES_DE_FONDO = True       # Habilita mezcla: intenta integrar coros de fondo (vocals atenuadas) en el acompañamiento
NIVEL_VOCES_DB = -21               # dB de reducción para atenuar la pista de vocales en la mezcla
# ==================================================

# Configurar logging para reducir verbosidad
logging.getLogger().setLevel(logging.ERROR)

# -------- Funciones Auxiliares --------

def extraer_audio_de_video(video_path: Path) -> Path | None:
    """Extrae la pista de audio de un video y la guarda como MP3."""
    print(f"Extrayendo audio de: {video_path.name}...")
    audio_path = video_path.with_suffix(".mp3")
    try:
        clip = VideoFileClip(str(video_path))
        if clip.audio is None:
            print(f"Advertencia: El video '{video_path.name}' no contiene pista de audio.")
            clip.close()
            return None
        clip.audio.write_audiofile(str(audio_path), codec='mp3', logger=None)
        clip.close()
        print(f"Audio extraído y guardado como: {audio_path.name}")
        return audio_path
    except Exception as e:
        print(f"Error al extraer audio del video '{video_path.name}': {e}")
        if 'clip' in locals() and clip:
            clip.close()
        return None

def separar_audio(audio_path: Path, output_dir: Path, model_name: str) -> Path | None:
    """Separa el audio usando Spleeter y devuelve el subdirectorio de salida generado."""
    print(f"Separando '{audio_path.name}' con Spleeter (Modelo: {model_name})...")
    try:
        output_dir.mkdir(parents=True, exist_ok=True)
        separator = Separator(model_name)
        separator.separate_to_file(str(audio_path), str(output_dir))
        output_subdir = output_dir / audio_path.stem
        if output_subdir.exists() and output_subdir.is_dir():
            print(f"Separación completada. Archivos generados en: {output_subdir}")
            return output_subdir
        else:
            print(f"Error: El directorio esperado '{output_subdir}' no fue creado por Spleeter.")
            return None
    except Exception as e:
        print(f"Error durante la separación de Spleeter para '{audio_path.name}': {e}")
        return None

def convertir_wav_a_mp3(wav_path: Path) -> Path | None:
    """Convierte un archivo WAV a MP3 usando moviepy."""
    if not wav_path.exists():
        print(f"Error: El archivo WAV '{wav_path.name}' no existe para la conversión.")
        return None
    mp3_path = wav_path.with_suffix(".mp3")
    print(f"Intentando convertir '{wav_path.name}' a MP3...")
    try:
        audio_clip = AudioFileClip(str(wav_path))
        audio_clip.write_audiofile(str(mp3_path), codec='mp3', logger=None)
        audio_clip.close()
        print(f"Conversión a MP3 exitosa: {mp3_path.name}")
        return mp3_path
    except Exception as e:
        print(f"Error al convertir '{wav_path.name}' a MP3: {e}")
        if 'audio_clip' in locals() and audio_clip:
            audio_clip.close()
        return None

# ================= Flujo Principal =================

print("Paso 1: Sube tus archivos (MP4 o MP3)")
archivos_subidos = files.upload()

if not archivos_subidos:
    print("No se subió ningún archivo.")
else:
    output_dir_general = Path(OUTPUT_DIR_NAME)
    output_dir_general.mkdir(parents=True, exist_ok=True)
    archivos_procesados = 0

    for nombre_original, contenido in archivos_subidos.items():
        print(f"\n--- Procesando Archivo: {nombre_original} ---")
        # Guardar el archivo subido de forma temporal
        temp_path = Path(nombre_original)
        temp_path.write_bytes(contenido)

        audio_file_to_process = None
        file_to_delete_later = []  # Para rastrear archivos temporales a eliminar

        extension = temp_path.suffix.lower()
        if extension == ".mp4":
            audio_file_to_process = extraer_audio_de_video(temp_path)
            if audio_file_to_process:
                file_to_delete_later.append(temp_path)         # El video original se elimina luego
                file_to_delete_later.append(audio_file_to_process)  # El audio extraído es temporal
            else:
                if temp_path.exists():
                    temp_path.unlink()
                continue
        elif extension == ".mp3":
            audio_file_to_process = temp_path
            file_to_delete_later.append(audio_file_to_process)
        else:
            print(f"Formato '{extension}' no soportado para '{nombre_original}'.")
            if temp_path.exists():
                temp_path.unlink()
            continue

        if audio_file_to_process and audio_file_to_process.exists():
            spleeter_output_subdir = separar_audio(audio_file_to_process, output_dir_general, SPLEETER_MODEL)
            if spleeter_output_subdir:
                accompaniment_wav = spleeter_output_subdir / "accompaniment.wav"
                vocals_wav = spleeter_output_subdir / "vocals.wav"

                if accompaniment_wav.exists():
                    final_output: Path | None = None

                    # Si se habilita la mezcla y existe la pista de vocales, intentar mezclar coros atenuados con el acompañamiento
                    if MEZCLAR_VOCES_DE_FONDO and vocals_wav.exists():
                        print(f"Intentando mezclar coros de fondo (reducción de {NIVEL_VOCES_DB} dB) con el acompañamiento...")
                        try:
                            acompanamiento_audio = AudioSegment.from_wav(str(accompaniment_wav))
                            voces_audio = AudioSegment.from_wav(str(vocals_wav))
                            # Reducir el volumen de las voces para enfatizar el acompañamiento
                            voces_atenuadas = voces_audio + NIVEL_VOCES_DB
                            # Igualar duración de ambos audios
                            duracion_minima = min(len(acompanamiento_audio), len(voces_atenuadas))
                            acompanamiento_audio = acompanamiento_audio[:duracion_minima]
                            voces_atenuadas = voces_atenuadas[:duracion_minima]
                            # Superponer las voces atenuadas sobre el acompañamiento
                            mezcla_final = acompanamiento_audio.overlay(voces_atenuadas)
                            final_output = accompaniment_wav.parent / f"{accompaniment_wav.stem}_con_coros_fondo.mp3"
                            print(f"Exportando mezcla final como: {final_output.name}")
                            mezcla_final.export(str(final_output), format="mp3")
                        except Exception as e:
                            print(f"Error durante la mezcla con pydub: {e}")
                            traceback.print_exc()
                            final_output = accompaniment_wav  # Fallback: usar acompañamiento original
                    else:
                        # Sin mezcla, se usa la pista de acompañamiento tal cual
                        final_output = accompaniment_wav

                    # Si se requiere conversión a MP3 y el archivo final es WAV, se convierte
                    if CONVERT_TO_MP3 and final_output.suffix.lower() == ".wav":
                        converted_mp3 = convertir_wav_a_mp3(final_output)
                        if converted_mp3 and converted_mp3.exists():
                            # Se elimina el WAV original si la conversión fue exitosa
                            if final_output != converted_mp3 and final_output.exists():
                                try:
                                    final_output.unlink()
                                except Exception:
                                    pass
                            final_output = converted_mp3

                    if final_output and final_output.exists():
                        print(f"Descargando: {final_output.name}")
                        files.download(str(final_output))
                        archivos_procesados += 1
                    else:
                        print("Error: No se pudo encontrar el archivo final para descargar.")
                else:
                    print(f"Error: No se encontró 'accompaniment.wav' en '{spleeter_output_subdir}'.")
            else:
                print(f"La separación de Spleeter falló para '{audio_file_to_process.name}'.")
        else:
            print(f"No se pudo obtener un archivo de audio válido de '{nombre_original}'.")
            if temp_path.exists():
                temp_path.unlink()

        # Limpiar archivos temporales de este ciclo
        for f_path in file_to_delete_later:
            try:
                if f_path and f_path.exists():
                    f_path.unlink()
            except OSError as e:
                print(f"Advertencia: No se pudo eliminar el archivo temporal {f_path.name}: {e}")

    if archivos_procesados > 0:
        print(f"\n--- Proceso completado para {archivos_procesados} archivo(s). ---")
    else:
        print("\n--- No se procesó exitosamente ningún archivo. ---")


Paso 1: Sube tus archivos (MP4 o MP3)


Saving Video de WhatsApp 2025-04-13 a las 07.12.13_85d295cf.mp4 to Video de WhatsApp 2025-04-13 a las 07.12.13_85d295cf.mp4

--- Procesando Archivo: Video de WhatsApp 2025-04-13 a las 07.12.13_85d295cf.mp4 ---
Extrayendo audio de: Video de WhatsApp 2025-04-13 a las 07.12.13_85d295cf.mp4...
Audio extraído y guardado como: Video de WhatsApp 2025-04-13 a las 07.12.13_85d295cf.mp3
Separando 'Video de WhatsApp 2025-04-13 a las 07.12.13_85d295cf.mp3' con Spleeter (Modelo: spleeter:2stems)...
INFO:spleeter:File output_spleeter/Video de WhatsApp 2025-04-13 a las 07.12.13_85d295cf/vocals.wav written succesfully


INFO:spleeter:File output_spleeter/Video de WhatsApp 2025-04-13 a las 07.12.13_85d295cf/vocals.wav written succesfully


INFO:spleeter:File output_spleeter/Video de WhatsApp 2025-04-13 a las 07.12.13_85d295cf/accompaniment.wav written succesfully


INFO:spleeter:File output_spleeter/Video de WhatsApp 2025-04-13 a las 07.12.13_85d295cf/accompaniment.wav written succesfully


Separación completada. Archivos generados en: output_spleeter/Video de WhatsApp 2025-04-13 a las 07.12.13_85d295cf
Intentando mezclar coros de fondo (reducción de -21 dB) con el acompañamiento...
Exportando mezcla final como: accompaniment_con_coros_fondo.mp3
Descargando: accompaniment_con_coros_fondo.mp3


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


--- Proceso completado para 1 archivo(s). ---
